<h1>🎛️ Biofilter — Report: <code>pair_variants</code></h1>

Candidate variant × variant pairs whose genes share biology.

Replaces six reports that were one pipeline written in three eras. Three
stages: **place** the input on genes, **connect** those genes through a
shared pathway / disease / protein, **pair** the variants.

Section 3 is the one to read: `max_group_size` decides both the size and
the meaning of the answer.

### 1. Open a bundle

In [ ]:
from pathlib import Path

from biofilter import Biofilter

BUNDLE = None
REPORT = "pair_variants"

bf = Biofilter(bundle=BUNDLE, debug_mode=False) if BUNDLE else Biofilter(debug_mode=False)

_root = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".biofilter.toml").is_file()),
    Path.cwd(),
)
OUTPUT_DIR = _root / "notebooks" / "templates" / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

GENES = ["CHEK2", "SMARCB1", "NF2"]
print(bf.core.db_uri)

### 2. What can link two genes in this bundle

A *group* is the entity sitting between two genes: a pathway they share,
a disease both are implicated in, a protein both interact with. Ask the
bundle which kinds it actually has edges for — Gene Ontology has 38,092
entities and no relationships at all, so it cannot link anything.

In [ ]:
from biofilter.modules.report import Bundle

with Bundle.open(bf.core.db_uri.removeprefix("parquet://")) as bundle:
    groups = bundle.con.execute("""
        WITH ge AS (SELECT e.id FROM entities e JOIN entity_groups eg ON eg.id = e.group_id
                    WHERE eg.name = 'Genes'),
        link AS (
            SELECT r.entity_1_id AS grp, r.entity_2_id AS gene FROM entity_relationships r
            WHERE r.entity_2_id IN (SELECT id FROM ge) AND r.entity_1_id NOT IN (SELECT id FROM ge)
            UNION ALL
            SELECT r.entity_2_id, r.entity_1_id FROM entity_relationships r
            WHERE r.entity_1_id IN (SELECT id FROM ge) AND r.entity_2_id NOT IN (SELECT id FROM ge))
        SELECT eg.name AS group_type, count(DISTINCT l.grp) AS groups,
               max(gene_n) AS largest_group
        FROM link l
        JOIN entities e ON e.id = l.grp
        JOIN entity_groups eg ON eg.id = e.group_id
        JOIN (SELECT grp, count(DISTINCT gene) AS gene_n FROM link GROUP BY 1) d
          ON d.grp = l.grp
        GROUP BY 1 ORDER BY 2 DESC
    """).to_arrow_table().to_pandas()

groups

### 3. `max_group_size`, and why an empty result is not "no biology"

A pathway naming 1,500 genes links its members while saying almost
nothing about any of them. The default drops groups above 300 genes.

Run the three genes at the default and watch it return nothing — then
read why.

In [ ]:
empty = bf.report.run("pair_genes", input_data=GENES)

print(f"{len(empty.to_pandas())} rows")
empty.provenance["group_filter"]

The three pathways linking these genes have 1,231, 1,321 and 1,543
genes each. Excluding them is correct — and without that provenance block
the empty frame would read as a finding about biology rather than a
consequence of a parameter.

In [ ]:
# Raise the limit, or use a group type whose members are smaller.
for size in (300, 2000):
    out = bf.report.run("pair_genes", input_data=GENES,
                        max_group_size=size).to_pandas()
    print(f"  max_group_size={size:>5}  {len(out):>3} gene pairs")

by_protein = bf.report.run("pair_genes", input_data=GENES,
                           group_types=["Proteins"]).to_pandas()
print(f"  group_types=Proteins   {len(by_protein):>3} gene pairs")
by_protein[["gene_1_symbol", "gene_2_symbol", "group_support_count"]]

### 4. `membership` — one side from the input, or both

The only difference there ever was between `variant_modeling` (`both`)
and `snp_snp_model` (`either`). Not a cosmetic filter: `either` is
unbounded in a way `both` is not.

In [ ]:
for membership in ("both", "either"):
    out = bf.report.run(REPORT, input_data=GENES, group_types=["Proteins"],
                        membership=membership, max_pairs=50_000)
    df = out.to_pandas()
    partners = df["gene_2_symbol"].nunique()
    print(f"  membership={membership:<7} {len(df):>7,} pairs, "
          f"{partners:>5,} partner genes, "
          f"truncated={out.provenance['truncation']['applied']}")

Five seed genes reach 19,393 partner genes through proteins, which
is why `either` needs its caps and `both` mostly does not.

### 5. Naming a gene and naming a variant are different requests

Naming `CHEK2` asks for its variants. Naming one rsID asks for that
variant — not for the other 4,000 in the gene that contains it.

In [ ]:
pairs = bf.report.run(REPORT, input_data=GENES, group_types=["Proteins"])
df = pairs.to_pandas()

print(f"{len(df):,} pairs from gene names")
df[["input_1", "variant_1_key", "gene_1_symbol", "variant_1_from_input",
    "variant_2_key", "gene_2_symbol", "variant_2_from_input",
    "group_support_count"]].head(5)

In [ ]:
# The same report given variants instead: every side is a named variant.
with Bundle.open(bf.core.db_uri.removeprefix("parquet://")) as bundle:
    named = [r[0] for r in bundle.con.execute("""
        SELECT v.variant_key FROM variant_masters v
        JOIN entity_locations l ON l.build = 38 AND l.chromosome = v.chromosome
         AND v.position BETWEEN l.start_pos AND l.end_pos
        JOIN gene_masters gm ON gm.entity_id = l.entity_id
        WHERE gm.symbol IN ('CHEK2', 'SMARCB1', 'NF2') AND v.af_joint > 0.05
        ORDER BY v.af_joint DESC LIMIT 40
    """).fetchall()]

from_variants = bf.report.run(REPORT, input_data=named,
                              group_types=["Proteins"]).to_pandas()

print(f"{len(named)} variants in -> {len(from_variants):,} pairs")
print("every side was named by hand:",
      bool(from_variants.variant_1_from_input.all()
           and from_variants.variant_2_from_input.all()))

### 6. `max_variants_per_gene` — where the size is really decided

Pairs grow with the **square** of the variants per gene. Three chr22
genes paired in full are 48 million rows, and the query runs out of
memory before producing them.

The variants kept are the most common, because a pairwise interaction
test has no power on a rare one. A variant you named is never dropped.

In [ ]:
for cap in (10, 100, 300):
    out = bf.report.run(REPORT, input_data=GENES, group_types=["Proteins"],
                        max_variants_per_gene=cap).to_pandas()
    print(f"  max_variants_per_gene={cap:>4}  {len(out):>7,} pairs")

pairs.provenance["pairing"]

### 7. Support, and what it is not

`group_support_count` counts the distinct groups linking the two genes.
It is a weight for ranking candidates — not a p-value, and not evidence
of interaction. Change `max_group_size` and every count changes.

In [ ]:
for support in (1, 20, 60):
    out = bf.report.run(REPORT, input_data=GENES, group_types=["Proteins"],
                        min_group_support=support).to_pandas()
    print(f"  min_group_support={support:>3}  {len(out):>7,} pairs")

df.nlargest(5, "group_support_count")[
    ["gene_1_symbol", "gene_2_symbol", "group_support_count", "group_support_names"]
]

### 8. Gene pairs are a different question

Stage 2 on its own — which of these genes are related, and by what — is
`pair_genes`, a report of its own rather than a mode of this one. It also
expands a gene pair by a list **you** supply, which is what to reach for
when the variant to gene attachment comes from outside the bundle: a
colocalization, a fine-mapping, a curated assignment.

`pair_variants` derives that attachment from coordinates, which is right
for a coding variant and wrong for a regulatory one.

In [ ]:
partners = bf.report.run("pair_genes", input_data=["CHEK2"],
                         group_types=["Proteins"], membership="either").to_pandas()

print(f"{len(partners):,} partner genes for CHEK2")
partners[["gene_1_symbol", "gene_2_symbol", "gene_2_from_input",
          "group_support_count", "group_support_sources"]].head(8)

### 9. Export

In [ ]:
for path in pairs.write(OUTPUT_DIR / "pair_variants.csv"):
    print(path)

### 10. The same thing on the command line

```bash
biofilter report run --report-name pair_variants \\
    --input CHEK2 --input SMARCB1 --input NF2 \\
    --param group_types=Proteins \\
    --param max_group_size=300 \\
    --param membership=both \\
    --output pairs.csv
```